# 02 ? Search Deep Dive: Query Specificity, ZRR, and CTR
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Thoroughly analyze search behavior, testing whether the long-tail search underperformance observed in the SQL audit is robust to segmentation.

---
### Areas of Investigation:
1. Query length (token count) vs Zero-Result Rate (ZRR)
2. Query specificity tier (Branded, Broad, Long-tail specific) vs CTR and downstream Order Conversion
3. Query type interactions with platform, user type, and category


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
from exploratory_analysis import get_db_connection, two_proportion_z_test, calc_odds_ratio

con = get_db_connection()


## 1. Query Length vs Zero-Result Rate (ZRR)


In [ ]:
q_tokens = '''
WITH tokens AS (
    SELECT 
        ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text), ' ')) AS token_count,
        is_zero_result,
        has_pdp_click,
        reformulated_in_session
    FROM search_events
)
SELECT 
    CASE WHEN token_count >= 5 THEN '5+ Tokens' ELSE CAST(token_count AS VARCHAR) || ' Token(s)' END AS token_tier,
    COUNT(*) AS total_searches,
    ROUND(100.0 * SUM(CASE WHEN is_zero_result THEN 1 ELSE 0 END) / COUNT(*), 2) AS zrr_pct,
    ROUND(100.0 * SUM(CASE WHEN has_pdp_click THEN 1 ELSE 0 END) / COUNT(*), 2) AS ctr_pct,
    ROUND(100.0 * SUM(CASE WHEN reformulated_in_session THEN 1 ELSE 0 END) / COUNT(*), 2) AS ref_pct
FROM tokens
GROUP BY 1
ORDER BY 1
'''
df_tokens = con.execute(q_tokens).df()
print("Search Performance by Token Count:")
print(df_tokens.to_string())


## 2. Query Type Performance & Downstream Conversion


In [ ]:
q_qt = '''
WITH s_agg AS (
    SELECT 
        se.session_id,
        MIN(se.query_type) AS primary_query_type,
        MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS has_order
    FROM search_events se
    LEFT JOIN orders o ON se.session_id = o.session_id
    GROUP BY se.session_id
)
SELECT 
    primary_query_type,
    COUNT(*) AS search_sessions,
    SUM(has_order) AS orders,
    ROUND(100.0 * SUM(has_order) / COUNT(*), 2) AS search_to_order_pct
FROM s_agg
GROUP BY 1
ORDER BY search_to_order_pct DESC
'''
df_qt = con.execute(q_qt).df()
print("Search-to-Order Conversion by Primary Query Intent:")
print(df_qt.to_string())

# Statistical test: Branded vs Long-tail specific
b_ord, b_tot = df_qt.loc[df_qt['primary_query_type']=='branded', ['orders', 'search_sessions']].values[0]
l_ord, l_tot = df_qt.loc[df_qt['primary_query_type']=='long_tail_specific', ['orders', 'search_sessions']].values[0]
diff, z, p, ci = two_proportion_z_test(b_ord, b_tot, l_ord, l_tot)
print(f"\nBranded vs Long-Tail Order Conversion Z-Test: Diff = +{diff*100:.2f} pp, Z = {z:.2f}, p-value = {p:.2e}")


## 3. Segment Interactions: Does Long-Tail Underperformance Persist Across Platforms?


In [ ]:
q_interact = '''
SELECT 
    s.platform,
    se.query_type,
    COUNT(*) AS total_searches,
    ROUND(100.0 * SUM(CASE WHEN se.is_zero_result THEN 1 ELSE 0 END) / COUNT(*), 2) AS zrr_pct,
    ROUND(100.0 * SUM(CASE WHEN se.has_pdp_click THEN 1 ELSE 0 END) / COUNT(*), 2) AS ctr_pct
FROM search_events se
JOIN sessions s ON se.session_id = s.session_id
GROUP BY 1, 2
ORDER BY 1, 2
'''
df_int = con.execute(q_interact).df()
print("Platform x Query Type Interaction Matrix:")
print(df_int.to_string())


## 4. Visualizations
- Search ZRR vs Token Count
- Search CTR by Query Type
- Search-to-Order Conversion by Query Type


In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/02_search_zrr_vs_tokens.png'))
display(Image(filename='../reports/figures/03_search_ctr_by_query_type.png'))
display(Image(filename='../reports/figures/04_search_conversion_by_query_type.png'))
